# 01 — Bronze Data, Features, and Targets

This notebook introduces the public data and modeling contracts without training a model. It loads a small ticker subset from bronze storage, converts it to the canonical market frame, inspects the versioned feature and target specifications, and executes their configured builders.

Edit the parameters in the next cell to match tickers that already exist in your local bronze database.

In [ ]:
from datetime import date
from pathlib import Path

PROVIDER = "yfinance"
TICKERS = ("AAK.ST", "SAAB-B.ST", "VOLV-B.ST")
START_DATE = date(2018, 1, 1)
END_DATE = date(2025, 12, 31)

## Load bronze data

`resolve_database_engine()` uses the configured application database URL. The bronze loader returns identifiers as columns because it is a storage boundary.

In [ ]:
from swingtrader.data.bronze.loaders import load_bronze_daily_prices
from swingtrader.data.db import resolve_database_engine

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
database_url = f"sqlite+pysqlite:///{(repo_root / 'data' / 'swingtrader.sqlite').as_posix()}"
engine = resolve_database_engine(database_url=database_url)
bronze = load_bronze_daily_prices(
    engine=engine,
    provider=PROVIDER,
    tickers=TICKERS,
    start_date=START_DATE,
    end_date=END_DATE,
    columns=(
        "open",
        "high",
        "low",
        "close",
        "adjusted_close",
        "volume",
    ),
)
bronze.head()

## Establish the canonical market frame

Feature and target builders require one unique, sorted `MultiIndex` named `provider`, `ticker`, and `trading_date`. The identifiers must not also remain ordinary columns.

In [ ]:
INDEX_COLUMNS = ["provider", "ticker", "trading_date"]
prices = bronze.set_index(INDEX_COLUMNS).sort_index()

assert prices.index.names == INDEX_COLUMNS
assert prices.index.is_unique
assert prices.index.is_monotonic_increasing
prices.head()

## Calculate a standalone indicator

Indicators are reusable numerical operations and do not know about the model feature contract. This example calculates ATR from the price frame supplied by the caller; the model feature layer separately owns its adjustment and normalization policy.

In [ ]:
from swingtrader.indicators.volatility import atr

atr_14 = atr(prices.loc[:, ["high", "low", "close"]], length=14)
atr_14.tail()

## Inspect immutable contracts

A feature set contains ordered `FeatureBlockSpec` objects. The notebook creates a smaller `FeatureSetSpec` from selected catalog blocks so the walkthrough stays quick while preserving the real builders and parameters. A target set contains ordered `TargetFamilySpec` objects, while a `SupervisedTaskSpec` selects one output as the supervised target.

In [ ]:
from swingtrader.data.features import DEFAULT_FEATURE_SET, FeatureSetSpec
from swingtrader.modeling.datasets import (
    V1_PRIMARY_TASK,
    V1_TARGET_SET,
    V2_PRIMARY_TASK,
    V2_TARGET_SET,
)

feature_set: FeatureSetSpec = DEFAULT_FEATURE_SET.select(
    "returns",
    "trend",
    "volatility",
    "volume",
    name="notebook_ohlcv_features",
    version="1",
)

print(feature_set.identifier)
print(feature_set.digest)
print(V1_TARGET_SET.identifier, V1_PRIMARY_TASK.target_column)
print(V2_TARGET_SET.identifier, V2_PRIMARY_TASK.target_column)

{
    "feature_set": feature_set.to_manifest(),
    "first_feature_block": feature_set.blocks[0].to_manifest(),
    "first_v2_target_family": V2_TARGET_SET.families[0].to_manifest(),
}

> **Reproducibility note:** This direct, windowed calculation is intentionally pedagogical. Expanding and path-dependent feature blocks can change when earlier history is truncated. The canonical dataset builder therefore loads the full legitimate history prefix required by the feature contract before applying the requested cutoff.

## Execute the feature set

The canonical temporal-dataset constructor applies the configured blocks in order. The loop below makes that composition visible for onboarding and then selects the contract-ordered output columns.

In [ ]:
feature_result = prices.copy(deep=True)
for block in feature_set.blocks:
    feature_result = block.apply(feature_result)

features = feature_result.loc[:, feature_set.feature_columns]
features.tail()

In [ ]:
feature_summary = features.agg(["count", "mean", "std"]).T
feature_summary.head(10)

## Generate target versions

Target generation intentionally uses future observations. Missing terminal labels remain missing rather than being coerced into the negative class.

In [ ]:
from swingtrader.modeling.datasets import generate_v1_labels, generate_v2_labels

v1_labels = generate_v1_labels(prices)
v2_labels = generate_v2_labels(prices)

v1_labels.loc[:, V1_TARGET_SET.target_columns].tail(10)

In [ ]:
v2_labels.loc[:, V2_TARGET_SET.target_columns].tail(10)

## Check alignment

Features and targets preserve the canonical index. A later `TemporalDatasetBundle` filters to rows where the selected target and its resolution date are available while retaining feature missing values for split-aware preprocessing.

In [ ]:
assert features.index.equals(v1_labels.index)
assert features.index.equals(v2_labels.index)

{
    "market_rows": len(prices),
    "feature_columns": len(features.columns),
    "v1_target_columns": len(V1_TARGET_SET.target_columns),
    "v2_target_columns": len(V2_TARGET_SET.target_columns),
    "v1_primary_available": int(v1_labels[V1_PRIMARY_TASK.target_column].notna().sum()),
    "v2_primary_available": int(v2_labels[V2_PRIMARY_TASK.target_column].notna().sum()),
}

Continue with `02_temporal_dataset_and_splitting.ipynb` to compose the specifications into a canonical bundle and assign leakage-safe train, validation, and locked-test ranges.